In [ ]:
# Importing neccessary libraries

# Data manipulation and visualization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

# Text preprocessing
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('stopwords')

# Dataset handling
from datasets import Dataset
from imblearn.under_sampling import RandomUnderSampler

# Hugging Face
from transformers import (
    Trainer,
    TrainingArguments,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    pipeline
)

# LoRA
from peft import LoraConfig, get_peft_model

# Evaluation
from sklearn.metrics import accuracy_score, precision_recall_fscore_support



# Load the review data
reviews_df = pd.read_json('../data/raw/yelp_academic_dataset_review.json', lines=True, nrows=100000) # Load only the first 100,000 reviews for analysis. In the report, mention resource management and the decision to limit the dataset size for initial exploration.
reviews_df = reviews_df[['stars', 'text']] # Keep only the relevant columns

# Remove duplicates and drop empty rows from the dataset
reviews_df.drop_duplicates(inplace=True)
reviews_df = reviews_df.dropna(subset=['text'])
reviews_df = reviews_df[reviews_df['text'].str.strip() != '']


# Display the frequency of star ratings
star_counts = reviews_df['stars'].value_counts().sort_index()
print(star_counts)

# Display the basic information about the dataset
print(reviews_df.info())

# Display the first few review texts and their corresponding star ratings
print(reviews_df.head(10))  

# Define a function for plotting distributions per star rating
def plot_stars_distribution(dataframe):
    plt.figure(figsize=(12, 6))

    star_rating_counts = dataframe['stars'].value_counts().sort_index()

    plt.bar(star_rating_counts.index, star_rating_counts.values, color='skyblue')
    plt.xlabel('Star Rating')
    plt.ylabel('Frequency')
    plt.title('Distribution of Star Ratings')
    #plt.show()

# Call the function to plot the distribution of star ratings
plot_stars_distribution(reviews_df)


# Data Preprocessing: lowercasing, removing punctuation, stopwords, html artifacts, and tokenization
stopwords_set = set(stopwords.words('english')) # Convert the stopwords list to a set for faster lookup during preprocessing

# Remove "no" and "not" from the stopwords list to preserve negation in sentiment analysis, as they can significantly change the meaning of a review (e.g., "not good" vs "good")
stopwords_set.discard('no')
stopwords_set.discard('not')

# Define a preprocessing function to clean the review texts
def preprocess_text(text):

    # Ensure the input is a string
    if not isinstance(text, str):
        return "" # Replace NaNs and other invalid values with an empty string to prevent errors during processing
    
    text = text.lower() # Convert to lowercase
    text = re.sub(r"<.*?>", "", text) # Remove HTML tags
    text = re.sub(r"[^a-zA-Z\s]", "", text) # Remove special characters
    text = re.sub(r"\d+", "", text) # Remove numbers
    tokens = word_tokenize(text) # Tokenize the text into words
    tokens = [word for word in tokens if word not in stopwords_set] # Remove stopwords using a list comprehension

    cleaned_text = " ".join(tokens) # Join the tokens back into a single string
    return cleaned_text

# Apply the preprocessing function to the review texts
reviews_df['cleaned_text'] = reviews_df['text'].apply(preprocess_text) # Create a new column for the cleaned review texts


# Filter the dataset to create a subset of reviews with only 1-star ratings
one_star_reviews = reviews_df[reviews_df['stars'] == 1]

# Analogously, create a subset of reviews with only 5-star ratings
five_star_reviews = reviews_df[reviews_df['stars'] == 5]


one_star_reviews_cleaned = one_star_reviews['cleaned_text']
five_star_reviews_cleaned = five_star_reviews['cleaned_text']

In [ ]:
def get_word_counts(review_text):
    combined_review_text = " ".join(review_text)
    review_word_list = combined_review_text.split()
    word_counts = Counter(review_word_list)

    return review_word_list, word_counts

one_star_word_list, one_star_word_counts = get_word_counts(one_star_reviews_cleaned)
five_star_word_list, five_star_word_counts = get_word_counts(five_star_reviews_cleaned)

print(f"Most frequent words in 100 one star reviews: {one_star_word_counts.most_common(100)}")
print(f"Most frequent words in 100 five star reviews: {five_star_word_counts.most_common(100)}")

AttributeError: 'Counter' object has no attribute 'head'